In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import time
import gget
import pickle

from sklearn.metrics.cluster import adjusted_rand_score 
from itertools import combinations

import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sc.settings.verbosity = 3
# sc.logging.print_header()

08:46:32 - INFO - Old pandas version detected. Patching DataFrame.map to DataFrame.applymap
/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


# Load the fibroblast data

In [2]:
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/scfib/scfib.gene_raw_feature_bc_matrix/"

fib_adata = sc.read_10x_mtx(fpath)
fib_adata.obs['batch'] = 'fib'

print(f"Batch: Fibroblast")
print(f"\tTotal UMIs: {int(np.sum(fib_adata.X))}")
print(f"\tCells: {fib_adata.shape[0]}")
print(f"\tGenes: {fib_adata.shape[1]}")

--> This might be very slow. Consider passing `cache=True`, which enables much faster reading from a cache file.
Batch: Fibroblast
	Total UMIs: 106198152
	Cells: 8963
	Genes: 23635


# load and merge feature barcodes

In [3]:
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/sc_fib_tsb_rerun/feature_barcodes/fbc_map.csv"
df = pd.read_csv(fpath)

# Sum columns for each condition:
df['G1'] = df.loc[:, df.columns.str.endswith('G1')].sum(axis=1)
df['S'] = df.loc[:, df.columns.str.endswith('S')].sum(axis=1)
df['G2M'] = df.loc[:, df.columns.str.endswith('G2M')].sum(axis=1)

# create a new dataframe with only the cell barcode and the summed conditions:
df = df[['cell_barcode', 'G1', 'S', 'G2M']]

df = df.groupby('cell_barcode', as_index=False).sum()
df['phase'] = df[['G1', 'S', 'G2M']].idxmax(axis=1)
df['cell_barcode'] = df['cell_barcode'] + "-1"
print(f"{df.shape=}")
barcode_map = dict(zip(df['cell_barcode'].values, df['phase'].values))
df.head()

df.shape=(10715, 5)


,cell_barcode,G1,S,G2M,phase
0,AAACCAAAGGGTAGCA-1,0.0,1.0,8.0,G2M
1,AAACCAAAGTAAGGGT-1,2.0,0.0,0.0,G1
2,AAACCATTCAGGTAGG-1,2.0,0.0,0.0,G1
3,AAACCATTCCAGCCCT-1,3.0,9.0,0.0,S
4,AAACCATTCGTGACCG-1,5.0,1.0,0.0,G1


In [4]:
fib_adata.obs['phase'] = fib_adata.obs.index.map(barcode_map)
print(fib_adata.obs['phase'].value_counts(dropna=False).to_string())

phase
G1     5733
S      1689
G2M    1453
NaN      88


# Add some metadata

In [5]:
model_str = '95m'
params_file = "/home/cstansbu/git_repositories/geneformer_v2/config/model_params.yaml"

with open(params_file, 'r') as file:
    params = yaml.safe_load(file)

model = params['models'][model_str]
print(json.dumps(model, indent=2))

{
  "model_path": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/gf-12L-95M-i4096/",
  "gene_median": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/gene_median_dictionary_gc95M.pkl",
  "token_dictionary_file": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/token_dictionary_gc95M.pkl",
  "gene_mapping_file": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/ensembl_mapping_dict_gc95M.pkl",
  "model_input_size": 4096,
  "special_token": true
}


In [6]:
with open(model['gene_mapping_file'], 'rb') as f:  
    gene_map = pickle.load(f)

print(f"{len(gene_map)=}")

len(gene_map)=173697


In [7]:
fib_adata.obs['nnz'] = (fib_adata.X != 0).sum(axis=1).A1 
fib_adata.var['feature_id'] = fib_adata.var.index.map(gene_map)

fib_adata

AnnData object with n_obs × n_vars = 8963 × 23635
    obs: 'batch', 'phase', 'nnz'
    var: 'gene_ids', 'feature_types', 'feature_id'

In [8]:
fib_adata.obs.head()

,batch,phase,nnz
AAACCAAAGGGTAGCA-1,fib,G2M,4556
AAACCAAAGTAAGGGT-1,fib,G1,2799
AAACCATTCAGGTAGG-1,fib,G1,4540
AAACCATTCCAGCCCT-1,fib,S,3055
AAACCATTCGTGACCG-1,fib,G1,3222


In [9]:
fib_adata.var.head()

,gene_ids,feature_types,feature_id
A1BG,unknown_00000,Gene Expression,ENSG00000121410
A1BG-AS1,unknown_00001,Gene Expression,ENSG00000268895
A1CF,unknown_00002,Gene Expression,ENSG00000148584
A2M,unknown_00003,Gene Expression,ENSG00000175899
A2M-AS1,unknown_00004,Gene Expression,ENSG00000245105


# Write the file

In [10]:
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/anndata/cc_fibroblast.h5ad"
fib_adata.write(outpath)
fib_adata

AnnData object with n_obs × n_vars = 8963 × 23635
    obs: 'batch', 'phase', 'nnz'
    var: 'gene_ids', 'feature_types', 'feature_id'